<a href="https://colab.research.google.com/github/raz0208/Natural-Language-Processing-Practices/blob/main/TopicModelling/EmbeddingsAnalysis_TopicModelling_TTDefault.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Topic Modelling

## Semantic Signal Separation

In [ ]:
!pip install turftopic

In [ ]:
# Import required libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

# import model and turftopic libraries
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from turftopic import SemanticSignalSeparation
import plotly.express as px

In [ ]:
# Read and load dataset
dataset = pd.read_csv('gdb_abstract.csv')

# Show the datasets
### Abstract Embeddings Sample Dataset
print('Node Content:', dataset.shape)
print(dataset)

In [ ]:
# Extract only the 'abstract' column and drop others
abstracts = dataset['abstract'].dropna().reset_index(drop=True)

# Display a few samples to verify
print(abstracts)

In [ ]:
#ٍ Extract embedding from the original dataset by TurfTopic default model
encoder = SentenceTransformer("paraphrase-MiniLM-L12-v2")
embeddings = encoder.encode(abstracts, show_progress_bar=True)

In [ ]:
embeddings

In [ ]:
# Check the dimention of each eambeding
sample = embeddings[0]

sample.shape

In [ ]:
# Extract topics
model = SemanticSignalSeparation(4, encoder=encoder, random_state=42)
doc_topic_matrix = model.fit_transform(abstracts, embeddings=embeddings)

In [ ]:
# Show the result of topics name
model.print_topics(top_k=10)

In [ ]:
model.plot_concept_compass(0, 1)

In [ ]:
# Topic Namer library
from turftopic.namers import LLMTopicNamer

In [ ]:
# Costumize topic namer

prompt = (
  "  are naming topics for a machine learning research paper."
  "Given these keywords: {keywords}, return a short and descriptive topic title."
  "Limit to 2-3 words. Return only the topic name."
)

namer = LLMTopicNamer(
    model_name="HuggingFaceTB/SmolLM2-1.7B-Instruct",
    prompt_template=prompt,
    # max_keywords=10,
    # temperature=0
)

In [ ]:
# namer = LLMTopicNamer("HuggingFaceTB/SmolLM2-1.7B-Instruct")
model.rename_topics(namer)

In [ ]:
model.print_topics()

In [ ]:
# model.rename_topics({
#     0: "Topic0",
#     1: "Topic1",
#     2: "Topic2",
#     3: "Topic4",
# })

In [ ]:
model.print_topic_distribution("I am a socialist and I am concerned with the growing inequality in our societies. I'd like to see governments do more to prevent the exploitation of workers.")

In [ ]:
df = pd.DataFrame(doc_topic_matrix, columns=model.topic_names)

fig = px.scatter_matrix(df, dimensions=model.topic_names, color="Topic0", template="plotly_white")
fig = fig.update_traces(diagonal_visible=False, showupperhalf=False, marker=dict(opacity=0.6))
fig.show()